#Project Overview

This project implements an AI-powered legal document summarization system designed to help lawyers quickly review lengthy agreements such as NDAs and SLAs. The solution automatically generates concise, legally coherent summaries that highlight key contractual clauses, enabling faster document understanding and decision-making.

# Install required packages

In [ ]:

!pip install transformers==4.35.0 datasets==2.18.0 rouge-score==0.1.2 nltk==3.8.1 spacy==3.6.0 sumy==0.8.1 python-docx tqdm evaluate reportlab
!pip install evaluate
!pip install sumy
!pip install rouge_score
!pip install evaluate
!pip install evaluate
!pip install reportlab
!pip install python-docx
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1) Imports and settings

In [ ]:

import os, glob, json, math, shutil
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import nltk
import evaluate
from transformers import T5ForConditionalGeneration, T5TokenizerFast, DataCollatorForSeq2Seq
from transformers import Trainer, TrainingArguments
from datasets import Dataset
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from rouge_score import rouge_scorer
import spacy
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from docx import Document
import zipfile

# Ensure NLTK tokenizer is available
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    print("Punkt tokenizer data not found. Downloading punkt_tab...")
    nltk.download('punkt_tab')


Punkt tokenizer data not found. Downloading punkt_tab...


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# 2) Paths and user parameters

In [ ]:

DATA_DIR = "/content/drive/MyDrive/full_contract_txt"
MASTER_CSV = "/content/master_clauses.csv"
SQUAD_JSON = "/content/CUAD_v1.json"
OUT_DIR = "output_cuad_summaries"
os.makedirs(OUT_DIR, exist_ok=True)

# Number of contracts to process for the demo
N_CONTRACTS = 10

# Number of words per chunk for model input
CHUNK_WORDS = 700

# Fine-tuning parameters for T5-small
MODEL_NAME = "t5-small"
TRAIN_EPOCHS = 1
BATCH_SIZE = 2
LEARNING_RATE = 5e-5

# Summary generation length parameters
GEN_MAX_LEN = 200
GEN_MIN_LEN = 40


# 3) Load contract TXT files

In [ ]:

contract_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))
if len(contract_files) == 0:
    raise FileNotFoundError(f"No .txt files found in '{DATA_DIR}'.")

# Select only N_CONTRACTS for the demo
contract_files = contract_files[:N_CONTRACTS]
contracts = []
for f in contract_files:
    with open(f, "r", encoding="utf-8", errors="ignore") as fh:
        text = fh.read().strip()
        contracts.append({"filename": os.path.basename(f), "text": text})
print(f"Loaded {len(contracts)} contracts.")


Loaded 10 contracts.


# 4) Load master_clauses.csv

In [ ]:

if not os.path.exists(MASTER_CSV):
    print("Warning: master_clauses.csv not found. Clause coverage and proxy golds will not be available.")
    clauses_df = None
else:
    clauses_df = pd.read_csv(MASTER_CSV, low_memory=False)
    print("Loaded master_clauses.csv:", clauses_df.shape)


Loaded master_clauses.csv: (510, 83)


# 5) Build gold summaries from master CSV

In [ ]:

proxy_gold_map = {}
if clauses_df is not None:
    fname_col = clauses_df.columns[0]
    for idx, row in clauses_df.iterrows():
        docname = row[fname_col]
        texts = []
        for col in clauses_df.columns[1:]:
            val = row[col]
            if isinstance(val, str) and val.strip():
                texts.append(val.strip())
        if texts:
            proxy_gold_map[docname] = " ".join(texts)

# Map proxy golds to selected contracts
selected_proxy_golds = {}
for c in contracts:
    base = c["filename"]
    candidates = [base, base.replace(".txt",""), base.replace(".txt","").replace(".pdf","")]
    found = None
    for cand in candidates:
        if cand in proxy_gold_map:
            found = proxy_gold_map[cand]
            break
    if found:
        selected_proxy_golds[c["filename"]] = found

print(f"Proxy gold summaries available for {len(selected_proxy_golds)}/{len(contracts)} contracts.")


gold_qa_json = {
  "version": "aok_v1.0",
  "data": [
    {
      "title": "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT",
      "paragraphs": [
        {
          "context": "EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  \"Agreement\")  is made by and between Electric City Corp.,  a Delaware  corporation  (\"Company\")  and Electric City of Illinois LLC (\"Distributor\") this 7th day of September, 1999. ...",
          "qas": [
            {
              "id": "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name",
              "question": "Highlight the parts (if any) of this contract related to 'Document Name' that should be reviewed by a lawyer. Details: The name of the contract",
              "is_impossible": False,
              "answers": [{"text": "DISTRIBUTOR AGREEMENT", "answer_start": 44}]
            },
            {
              "id": "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties",
              "question": "Highlight the parts (if any) of this contract related to 'Parties' that should be reviewed by a lawyer. Details: The two or more parties who signed the contract",
              "is_impossible": False,
              "answers": [
                {"text": "Distributor", "answer_start": 244},
                {"text": "Electric City Corp.", "answer_start": 148},
                {"text": "Electric City of Illinois LLC", "answer_start": 49574},
                {"text": "Company", "answer_start": 197},
                {"text": "Electric City of Illinois LLC", "answer_start": 212}
              ]
            },
            {
              "id": "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Agreement Date",
              "question": "Highlight the parts (if any) of this contract related to 'Agreement Date' that should be reviewed by a lawyer. Details: The date of the contract",
              "is_impossible": False,
              "answers": [{"text": "7th day of September, 1999.", "answer_start": 263}]
            },
            {
              "id": "LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Governing Law",
              "question": "Highlight the parts (if any) of this contract related to 'Governing Law' that should be reviewed by a lawyer. Details: Which state/country's law governs the interpretation of the contract?",
              "is_impossible": False,
              "answers": [
                {"text": "This Agreement is to be construed according to the laws of the State of Illinois.", "answer_start": 52061}
              ]
            }
          ]
        }
      ]
    }
  ]
}
# Convert QA JSON to combined gold summaries
gold_qa_summaries = {}
for entry in gold_qa_json["data"]:
    title = entry["title"]
    all_texts = []
    for para in entry["paragraphs"]:
        for qa in para["qas"]:
            for ans in qa["answers"]:
                all_texts.append(ans["text"])
    if all_texts:
        gold_qa_summaries[f"{title}.txt"] = " | ".join(all_texts)

# Merge QA-based golds into proxy map
selected_proxy_golds.update(gold_qa_summaries)
print(f"Added {len(gold_qa_summaries)} gold summaries from inline QA JSON.")


Proxy gold summaries available for 0/10 contracts.
Added 1 gold summaries from inline QA JSON.


# 6) Preprocessing and chunking of contracts

In [ ]:

def chunk_text_words(text, max_words=CHUNK_WORDS):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunks.append(" ".join(words[i:i+max_words]))
    return chunks

for c in contracts:
    c["chunks"] = chunk_text_words(c["text"], CHUNK_WORDS)

# 7) Prepare training dataset for fine-tuning

In [ ]:

train_examples = []
val_examples = []
for c in contracts:
    if c["filename"] in selected_proxy_golds:
        input_text = "summarize: " + " ".join(c["chunks"][:3])
        target = selected_proxy_golds[c["filename"]][:1000]
        train_examples.append({"text": input_text, "summary": target})

if len(train_examples) < 2:
    print("Not enough proxy-gold examples for fine-tuning demo. Will use pretrained T5-small for inference.")
else:
    split = int(0.8 * len(train_examples))
    train_set = train_examples[:split]
    val_set = train_examples[split:]
    print(f"Prepared {len(train_set)} train and {len(val_set)} val examples.")

Not enough proxy-gold examples for fine-tuning demo. Will use pretrained T5-small for inference.


# 8) Initialize tokenizer and model

In [ ]:

tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# 9) Fine-tune T5

In [ ]:

def prepare_dataset_for_trainer(examples):
    inputs = tokenizer([ex["text"] for ex in examples], truncation=True, padding="longest", max_length=1024)
    labels = tokenizer([ex["summary"] for ex in examples], truncation=True, padding="longest", max_length=256)
    inputs["labels"] = labels["input_ids"]
    return Dataset.from_dict(inputs)

if len(train_examples) >= 2:
    hf_train = prepare_dataset_for_trainer(train_set)
    hf_val = prepare_dataset_for_trainer(val_set)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, "t5_finetune"),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        predict_with_generate=True,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        num_train_epochs=TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=False,
        push_to_hub=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    print("Starting fine-tuning (demo).")
    trainer.train()
    print("Fine-tuning complete.")

# 10) Generate abstractive summaries

In [ ]:

summaries_abstr = []
model.eval()
for c in tqdm(contracts):
    chunk_summaries = []
    for ch in c["chunks"]:
        input_ids = tokenizer("summarize: " + ch, return_tensors="pt", truncation=True, padding="longest", max_length=1024).input_ids
        out_ids = model.generate(input_ids, max_length=GEN_MAX_LEN, min_length=GEN_MIN_LEN, num_beams=2, early_stopping=True)
        s = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        chunk_summaries.append(s)
    full_summary = " ".join(chunk_summaries)
    summaries_abstr.append({"filename": c["filename"], "summary": full_summary})

100%|██████████| 10/10 [09:58<00:00, 59.84s/it]


# 11) Generate extractive summaries (LexRank)

In [ ]:

summaries_extr = []
for c in tqdm(contracts):
    parser = PlaintextParser.from_string(c["text"], Tokenizer("english"))
    summarizer_lex = LexRankSummarizer()
    sentences = summarizer_lex(parser.document, sentences_count=6)
    extractive_text = " ".join(str(s) for s in sentences)
    summaries_extr.append({"filename": c["filename"], "summary": extractive_text})


100%|██████████| 10/10 [00:14<00:00,  1.41s/it]


# 12) Evaluate summaries using ROUGE

In [ ]:

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
eval_rows = []
for i,c in enumerate(contracts):
    fn = c["filename"]
    abstr = summaries_abstr[i]["summary"]
    extr = summaries_extr[i]["summary"]
    proxy = selected_proxy_golds.get(fn, None)
    if proxy is None:
        r_abstr_vs_extr = scorer.score(abstr, extr)
        eval_rows.append({
            "filename": fn,
            "rouge1_abstr_vs_extr": r_abstr_vs_extr['rouge1'].fmeasure,
            "rouge2_abstr_vs_extr": r_abstr_vs_extr['rouge2'].fmeasure,
            "rougeL_abstr_vs_extr": r_abstr_vs_extr['rougeL'].fmeasure,
            "proxy_exists": False
        })
    else:
        r_abstr = scorer.score(abstr, proxy)
        r_extr = scorer.score(extr, proxy)
        eval_rows.append({
            "filename": fn,
            "rouge1_abstr_vs_proxy": r_abstr['rouge1'].fmeasure,
            "rouge2_abstr_vs_proxy": r_abstr['rouge2'].fmeasure,
            "rougeL_abstr_vs_proxy": r_abstr['rougeL'].fmeasure,
            "rouge1_extr_vs_proxy": r_extr['rouge1'].fmeasure,
            "rouge2_extr_vs_proxy": r_extr['rouge2'].fmeasure,
            "rougeL_extr_vs_proxy": r_extr['rougeL'].fmeasure,
            "proxy_exists": True
        })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(os.path.join(OUT_DIR, "evaluation_scores.csv"), index=False)
print("Saved evaluation_scores.csv")

Saved evaluation_scores.csv


# 13) Clause coverage evaluation (keyword-based heuristic)

In [ ]:
clause_keywords = {
    "Confidentiality": ["confidential", "confidentiality", "non-disclosure", "nda"],
    "Indemnity": ["indemnif", "hold harmless", "indemnity"],
    "Termination": ["terminate", "termination", "terminate for convenience", "terminate for cause"],
    "Governing Law": ["governed by", "governing law", "laws of"],
    "Effective Date": ["effective date", "effective as of", "effective"],
    "Expiration Date": ["expire", "expiration", "term", "end of term"],
    "Insurance": ["insurance", "insured", "coverage"],
    "Non-Compete": ["non-compete", "non compete"]
}


coverage_rows = []
for i,c in enumerate(contracts):
    fn = c["filename"]
    abstr = summaries_abstr[i]["summary"].lower()
    extr = summaries_extr[i]["summary"].lower()
    proxy = selected_proxy_golds.get(fn, "")
    proxy_lower = proxy.lower() if proxy else ""
    row = {"filename": fn}
    for clause, kws in clause_keywords.items():
        in_proxy = any(k in proxy_lower for k in kws)
        in_abstr = any(k in abstr for k in kws)
        in_extr = any(k in extr for k in kws)
        row[f"{clause}_in_proxy"] = in_proxy
        row[f"{clause}_in_abstr"] = in_abstr
        row[f"{clause}_in_extr"] = in_extr
    coverage_rows.append(row)
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(os.path.join(OUT_DIR, "clause_coverage.csv"), index=False)
print("Saved clause_coverage.csv")

Saved clause_coverage.csv


# 14) Export first 5 abstractive summaries to PDFs

In [ ]:

pdf_dir = os.path.join(OUT_DIR, "summaries_pdf")
os.makedirs(pdf_dir, exist_ok=True)
for i, s in enumerate(summaries_abstr[:5]):
    pdf_path = os.path.join(pdf_dir, f"{s['filename']}_summary.pdf")
    c = canvas.Canvas(pdf_path, pagesize=letter)
    text_obj = c.beginText(40, 750)
    text_obj.setFont("Times-Roman", 11)
    text_obj.textLine(f"Contract: {s['filename']}")
    text_obj.textLine("")
    for para in s['summary'].split(". "):
        text_obj.textLine(para.strip())
        if text_obj.getY() < 40:
            c.drawText(text_obj)
            c.showPage()
            text_obj = c.beginText(40, 750)
            text_obj.setFont("Times-Roman", 11)
    c.drawText(text_obj)
    c.save()

print(f"Saved {min(5,len(summaries_abstr))} PDF summaries to {pdf_dir}")


Saved 5 PDF summaries to output_cuad_summaries/summaries_pdf


# 15) Generate DOCX report comparing Extractive vs Abstractive

In [ ]:

doc = Document()
doc.add_heading('Extractive vs Abstractive Summarization Report', level=1)
doc.add_paragraph('Project: Legal Document Summarization using CUAD (CPU demo).')
doc.add_paragraph(f'Number of contracts processed: {len(contracts)}')

doc.add_heading('Evaluation (ROUGE)', level=2)
doc.add_paragraph('Saved evaluation table: evaluation_scores.csv')
doc.add_paragraph('Key results (per-contract):')
for idx, row in eval_df.head(10).iterrows():
    doc.add_paragraph(str(row.to_dict()))

doc.add_heading('Clause Coverage (heuristic)', level=2)
doc.add_paragraph('Saved clause coverage table: clause_coverage.csv')
doc.add_paragraph('Example (first rows):')
doc.add_paragraph(coverage_df.head(5).to_string())

doc_path = os.path.join(OUT_DIR, "extractive_vs_abstractive_report.docx")
doc.save(doc_path)
print("Saved report:", doc_path)


Saved report: output_cuad_summaries/extractive_vs_abstractive_report.docx


# 16) Save model for reuse

In [ ]:

save_model_dir = os.path.join(OUT_DIR, "t5_model_saved")
model.save_pretrained(save_model_dir)
tokenizer.save_pretrained(save_model_dir)
print("Saved model to:", save_model_dir)

Saved model to: output_cuad_summaries/t5_model_saved


# 17) Package submission ZIP

In [ ]:

zip_path = os.path.join(OUT_DIR, "submission_package.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(os.path.join(OUT_DIR, "evaluation_scores.csv"), arcname="evaluation_scores.csv")
    zf.write(os.path.join(OUT_DIR, "clause_coverage.csv"), arcname="clause_coverage.csv")
    zf.write(doc_path, arcname=os.path.basename(doc_path))
    for f in os.listdir(pdf_dir):
        zf.write(os.path.join(pdf_dir, f), arcname=os.path.join("summaries_pdf", f))
print("Created submission zip:", zip_path)

Created submission zip: output_cuad_summaries/submission_package.zip
